In [1]:
!pip -q install -U datasets transformers peft trl accelerate bitsandbytes huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.5/527.5 kB 11.6 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 74.0 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 28.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 612.9/612.9 kB 36.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 82.2 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 38.4 MB/s eta 0:00:00:00:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.31.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.21.0 requires google-cloud-bigquery

In [2]:
import torch
print(torch.cuda.is_available())

True


In [5]:
%%writefile prepare_dataset.py
from datasets import load_dataset, Dataset, DatasetDict
import json
import os

# Load TriviaQA (answerable questions)
print("Loading TriviaQA...")
trivia = load_dataset("mandarjoshi/trivia_qa", "rc", split="train")

# Load SQuAD 2.0 (unanswerable questions)
print("Loading SQuAD 2.0...")
squad = load_dataset("rajpurkar/squad_v2", split="train")

#choose the correct context paragraph for each question/answer. if non match, return None
def pick_context_with_answer(contexts, answer):
    ans = answer.strip().lower()
    for c in contexts:
        if c and ans in c.lower():
            return c
    return None


def shrink_context_around_answer(context: str, answer: str, max_chars: int = 4000, min_window: int = 800):
    """
    Returns a substring of `context` centered around the first occurrence of `answer`,
    with length <= max_chars, attempting to keep sentence boundaries.
    If answer isn't found, returns None.

    max_chars: final cap for context length (characters)
    min_window: if context is already short, keep it (helps avoid over-cropping)
    """
    if not context or not answer:
        return None

    ctx_lower = context.lower()
    ans_lower = answer.strip().lower()
    idx = ctx_lower.find(ans_lower)
    if idx == -1:
        return None

    # If already small enough, keep as-is
    if len(context) <= max_chars:
        return context

    # Center window around answer span
    ans_end = idx + len(answer)
    center = (idx + ans_end) // 2

    half = max_chars // 2
    start = max(0, center - half)
    end = min(len(context), center + half)

    snippet = context[start:end]

    # Try to align to sentence-ish boundaries to look nicer
    # (simple heuristics; good enough for this project)
    if start > 0:
        # move start forward to the next likely boundary
        cut = max(snippet.find(". "), snippet.find("\n"))
        if cut != -1 and cut < 300:  # don't cut too aggressively
            snippet = snippet[cut+2:] if snippet[cut:cut+2] == ". " else snippet[cut+1:]

    if end < len(context):
        # move end backward to last boundary in snippet
        last_period = snippet.rfind(". ")
        last_nl = snippet.rfind("\n")
        cut = max(last_period, last_nl)
        if cut != -1 and (len(snippet) - cut) < 300:
            snippet = snippet[:cut+1]

    # If we somehow over-cropped too hard, ensure at least min_window if possible
    if len(snippet) < min_window and len(context) > min_window:
        # expand a bit around original center
        half2 = min_window // 2
        start2 = max(0, center - half2)
        end2 = min(len(context), center + half2)
        snippet = context[start2:end2]

    return snippet.strip()

def format_trivia(example):
    # Grab first wikipedia context passage
    contexts = example["entity_pages"]["wiki_context"]
    context = pick_context_with_answer(contexts, example["answer"]["value"])
    if not context:
        return None

    question = example["question"]
    answer = example["answer"]["value"]
    
    context = shrink_context_around_answer(context, answer, max_chars=4000)
    if not context:
        return None
    
    return {
        "prompt": f"Context: {context}\nQuestion: {question}\nAnswer:",
        "response": f"FINAL: {answer}",
        "label": 1,
    }
    




def format_squad_unanswerable(example):
    # SQuAD 2.0 has empty answers for unanswerable questions
    if len(example["answers"]["text"]) == 0:
        return {
            "prompt": f"Context: {example['context']}\nQuestion: {example['question']}\nAnswer:",
            "response": "FINAL: I don't know.",
            "label": 0,
        }
    return None


# Format and filter
print("Formatting TriviaQA...")
trivia_formatted = []
for ex in trivia.select(range(50000)):  # try more because of filtering
    r = format_trivia(ex)
    if r:
        trivia_formatted.append(r)
    if len(trivia_formatted) >= 20000:
        break
trivia_formatted = [ex for ex in trivia_formatted if ex is not None]

print("Formatting SQuAD unanswerables...")
squad_unanswerable = []
for ex in squad:
    result = format_squad_unanswerable(ex)
    if result:
        squad_unanswerable.append(result)
    if len(squad_unanswerable) >= 7000:
        break

# Combine
full_dataset = trivia_formatted + squad_unanswerable
print(f"Total examples: {len(full_dataset)}")
print(f"Answerable: {len(trivia_formatted)}, Unanswerable: {len(squad_unanswerable)}")


ds = Dataset.from_list(full_dataset)

# Deterministic shuffle
SEED = 42
ds = ds.shuffle(seed=SEED)

# Train/val/test split (80/10/10)
ds_splits = ds.train_test_split(test_size=0.2, seed=SEED)          # 80/20
tmp = ds_splits["test"].train_test_split(test_size=0.5, seed=SEED) # 10/10
ds_dict = DatasetDict({
    "train": ds_splits["train"],
    "val": tmp["train"],
    "test": tmp["test"],
})

print(ds_dict)
print({k: len(v) for k, v in ds_dict.items()})

# Preview one of each
print("\n--- Train example ---")
print(ds_dict["train"][0])

# ---- Save to disk (recommended) ----
os.makedirs("data/hf_dataset", exist_ok=True)
ds_dict.save_to_disk("data/hf_dataset")
print("Saved HF DatasetDict to data/hf_dataset")

Overwriting prepare_dataset.py


In [6]:
!python prepare_dataset.py

Loading TriviaQA...
README.md: 26.7kB [00:00, 62.9MB/s]
Resolving data files: 100%|██████████████████| 26/26 [00:00<00:00, 25958.56it/s]
rc/train-00000-of-00026.parquet:   0%|               | 0.00/308M [00:00<?, ?B/s]
rc/train-00000-of-00026.parquet:   0%|               | 0.00/308M [00:00<?, ?B/s]
rc/train-00000-of-00026.parquet:   0%|               | 0.00/308M [00:00<?, ?B/s]
rc/train-00000-of-00026.parquet:   0%|               | 0.00/308M [00:00<?, ?B/s]
rc/train-00000-of-00026.parquet:   0%|               | 0.00/308M [00:00<?, ?B/s]
rc/train-00000-of-00026.parquet:   0%|       | 39.3k/308M [00:01<25:59, 197kB/s]
rc/train-00000-of-00026.parquet:   0%|       | 96.5k/308M [00:01<32:38, 157kB/s]
rc/train-00000-of-00026.parquet:   1%|      | 2.22M/308M [00:01<01:20, 3.77MB/s]
rc/train-00000-of-00026.parquet:   2%|▏     | 6.62M/308M [00:02<00:40, 7.36MB/s]
rc/train-00000-of-00026.parquet:   4%|▏     | 12.1M/308M [00:02<00:23, 12.7MB/s]
rc/train-00000-of-00026.parquet:   7%|▍     | 20.4M/3

In [7]:
!rm -rf ./data/qa_abstain_dataset_clipped
!rm -f ./data/abstain_train.jsonl
!rm -rf ./outputs_abstain_fast
!rm -rf ./outputs_abstain
!rm -rf ./adapter_abstain*

In [8]:
from huggingface_hub import login
# login()   # uncomment to login

In [19]:
import os, json, re, random
import torch

from datasets import load_dataset, load_from_disk
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    BitsAndBytesConfig, TrainingArguments
)
from peft import LoraConfig, get_peft_model, PeftModel
from trl import SFTTrainer

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

ABSTAIN_STR = "FINAL: I don't know."

DATA_DIR = "./data"
JSONL_PATH = os.path.join(DATA_DIR, "abstain_train.jsonl")
CACHE_DIR = os.path.join(DATA_DIR, "qa_abstain_dataset_clipped")

# Prevent RAM blowups: keep only first N chars of each context
MAX_CHARS = 2000  # reduce to 1200 if you still crash later / want faster training

# Dataset sizes (tune as needed)
N_TRIVIA_TARGET = 15000   # realistic on streaming filter; bump if you want more
N_UNANS_TARGET  = 7000

os.makedirs(DATA_DIR, exist_ok=True)

def normalize(s: str) -> str:
    return re.sub(r"\s+", " ", (s or "").strip().lower())

def clip(text: str, max_chars=MAX_CHARS) -> str:
    t = (text or "").strip()
    return t[:max_chars]

In [11]:
def get_gold_from_text(text: str) -> str:
    # gold is always the last line like "FINAL: ..."
    last_line = text.strip().splitlines()[-1].strip()
    return last_line  # includes "FINAL: ..."

In [12]:
# Load from pre-created HuggingFace dataset folder
HF_DATASET_PATH = "/kaggle/working/data/hf_dataset"
print(f"Loading datasets from: {HF_DATASET_PATH}")

# Load the train, val, and test splits
train_ds = load_from_disk(os.path.join(HF_DATASET_PATH, "train"))
eval_ds = load_from_disk(os.path.join(HF_DATASET_PATH, "val"))
test_ds = load_from_disk(os.path.join(HF_DATASET_PATH, "test"))

print(f"Train: {len(train_ds)}, Val: {len(eval_ds)}, Test: {len(test_ds)}")
print("Sample train data:", train_ds[0])

Loading datasets from: /kaggle/working/data/hf_dataset
Train: 21600, Val: 2700, Test: 2700
Sample train data: {'prompt': "Context: New Delhi   is the capital and seat of the Government of India. It is also a municipality and district in Delhi and serves as the seat of Government of Delhi.\n\nThe foundation stone of the city was laid by George V, Emperor of India during the Delhi Durbar of 1911.  It was designed by British architects, Sir Edwin Lutyens and Sir Herbert Baker. The new capital was inaugurated on 13 February 1931,  by Viceroy and Governor-General of India Lord Irwin.\n\nAlthough colloquially Delhi and New Delhi as names are used interchangeably to refer to the jurisdiction of the National Capital Territory (NCT) of Delhi, these are two distinct entities, and the latter is a small part of the former.\n\nNew Delhi has been selected as one of the hundred Indian cities to be developed as a smart city under PM Narendra Modi's flagship Smart Cities Mission.\n\nHistory\n\nEstablis

In [13]:
# Safe default (open + small)
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

# If you have access and want bigger:
# MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"
# MODEL_NAME = "meta-llama/Llama-2-7b-chat-hf"  # requires access
print("Using model:", MODEL_NAME)

Using model: Qwen/Qwen2.5-1.5B-Instruct


In [14]:
# Datasets are already loaded from the HuggingFace dataset folder
# Dataset format has 'prompt' and 'response' fields instead of combined 'text'

print("Train:", len(train_ds), "Val:", len(eval_ds), "Test:", len(test_ds))
print("\nDataset fields:", train_ds.column_names)
print("\nSample from train dataset:")
sample = train_ds[0]
print(sample)
print("\nPrompt (first 250 chars):")
print(sample["prompt"][:250])
print("\nResponse:")
print(sample["response"])

Train: 21600 Val: 2700 Test: 2700

Dataset fields: ['prompt', 'response', 'label']

Sample from train dataset:
{'prompt': "Context: New Delhi   is the capital and seat of the Government of India. It is also a municipality and district in Delhi and serves as the seat of Government of Delhi.\n\nThe foundation stone of the city was laid by George V, Emperor of India during the Delhi Durbar of 1911.  It was designed by British architects, Sir Edwin Lutyens and Sir Herbert Baker. The new capital was inaugurated on 13 February 1931,  by Viceroy and Governor-General of India Lord Irwin.\n\nAlthough colloquially Delhi and New Delhi as names are used interchangeably to refer to the jurisdiction of the National Capital Territory (NCT) of Delhi, these are two distinct entities, and the latter is a small part of the former.\n\nNew Delhi has been selected as one of the hundred Indian cities to be developed as a smart city under PM Narendra Modi's flagship Smart Cities Mission.\n\nHistory\n\nEstabli

In [15]:
ex = train_ds[0]
prompt = ex["prompt"]
response = ex["response"]
gold = response.replace("FINAL:", "", 1).strip()

# Extract context from the prompt
ctx = prompt.split("Context:",1)[1].split("Question:",1)[0] if "Context:" in prompt else ""
print("Prompt length:", len(prompt))
print("Gold:", gold)
print("Gold in context?", gold.lower() in ctx.lower() if ctx else "N/A")

Prompt length: 2245
Gold: Edwin LUTYENS
Gold in context? True


In [20]:
use_cuda = torch.cuda.is_available()
print("CUDA available:", use_cuda)
if use_cuda:
    !nvidia-smi

# Try QLoRA (4-bit). If it fails, we fall back to normal LoRA.
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

try:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        device_map="auto",
        quantization_config=bnb_config,
    )
    print("Loaded model in 4-bit (QLoRA-ready).")
    quant_mode = "4bit"
except Exception as e:
    print("4-bit load failed, falling back to full precision LoRA.\nReason:", repr(e))
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        device_map="auto",
        torch_dtype=torch.float16 if use_cuda else torch.float32,
    )
    quant_mode = "fp16/fp32"

CUDA available: True
Tue Mar 10 04:05:17 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   55C    P0             27W /   70W |     623MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+--------------------------

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Loaded model in 4-bit (QLoRA-ready).


In [21]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


In [37]:
def combine_prompt_response(example):
    """Combine prompt and response into a 'text' field for training"""
    example["text"] = example["prompt"] +"\n" + example["response"]
    return example

def rename_response_to_completion(example):
    """Rename 'response' field to 'completion' for SFTTrainer compatibility"""
    example["completion"] = example["response"]
    return example

# Convert label from int to string for compatibility
def convert_label(example):
    """Convert numeric label to string label"""
    example["label"] = "answerable" if example["label"] == 1 else "unanswerable"
    return example

# Apply transformations to datasets
train_ds = train_ds.map(combine_prompt_response).map(rename_response_to_completion).map(convert_label)
eval_ds = eval_ds.map(combine_prompt_response).map(rename_response_to_completion).map(convert_label)
test_ds = test_ds.map(combine_prompt_response).map(rename_response_to_completion).map(convert_label)

def split_text_to_prompt_resp(text: str):
    """Extract prompt and response from combined text field"""
    # Find where the response starts (after "Answer:\n")
    parts = text.rsplit("Answer:\n", 1)
    if len(parts) == 2:
        prompt = parts[0] + "Answer:\n"
        resp = parts[1]
    else:
        # Fallback: last line is response
        lines = text.strip().splitlines()
        resp = lines[-1].strip()
        prompt = "\n".join(lines[:-1]).strip() + "\n"
    return prompt, resp

def extract_final_answer(s: str) -> str:
    m = re.search(r"FINAL:\s*(.*)", s, flags=re.IGNORECASE)
    return m.group(1).strip() if m else s.strip()

Map:   0%|          | 0/21600 [00:00<?, ? examples/s]

Map:   0%|          | 0/21600 [00:00<?, ? examples/s]

Map:   0%|          | 0/21600 [00:00<?, ? examples/s]

Map:   0%|          | 0/2700 [00:00<?, ? examples/s]

Map:   0%|          | 0/2700 [00:00<?, ? examples/s]

Map:   0%|          | 0/2700 [00:00<?, ? examples/s]

Map:   0%|          | 0/2700 [00:00<?, ? examples/s]

Map:   0%|          | 0/2700 [00:00<?, ? examples/s]

Map:   0%|          | 0/2700 [00:00<?, ? examples/s]

In [23]:
print("Combined text format (first 300 chars):")
print(train_ds[0]["text"][:300])

Combined text format (first 300 chars):
Context: New Delhi   is the capital and seat of the Government of India. It is also a municipality and district in Delhi and serves as the seat of Government of Delhi.

The foundation stone of the city was laid by George V, Emperor of India during the Delhi Durbar of 1911.  It was designed by Britis


In [24]:
p, g = split_text_to_prompt_resp(eval_ds[0]["text"])
print("PROMPT tail (last 200 chars):", p[-200:])
print("\nGOLD:", g)

PROMPT tail (last 200 chars):  Chart in 1953) 
*Jimmy Roselli - 1967
*Mel Blanc - N/k
Question: "Which British bandleader who lived from 1899 to 1969 would you associate with the song ""Somebody Stole My Gal"" recorded in 1931 ?"


GOLD: Answer:FINAL: BILLY COTTON


In [25]:
prompt, gold_resp = split_text_to_prompt_resp(eval_ds[0]["text"])
gold = extract_final_answer(gold_resp)

# extract context from the prompt
ctx = prompt.split("Context:",1)[1].split("Question:",1)[0] if "Context:" in prompt else ""
print("Gold:", gold)
print("Gold in context?", gold.lower() in ctx.lower() if ctx else "N/A")

Gold: BILLY COTTON
Gold in context? True


In [26]:


from trl import SFTTrainer, SFTConfig
from datasets import concatenate_datasets
import torch

# --- small subset ---
# from datasets import concatenate_datasets

ans = train_ds.filter(lambda x: x["label"]=="answerable").shuffle(seed=42).select(range(4750))
unans = train_ds.filter(lambda x: x["label"]=="unanswerable").shuffle(seed=42).select(range(250))
train_small = concatenate_datasets([ans, unans]).shuffle(seed=42)
eval_small = eval_ds.shuffle(seed=42).select(range(200))

# --- stability ---
model.gradient_checkpointing_disable()
model.config.use_cache = True

training_args = SFTConfig(
    output_dir="./outputs_abstain_10min",
    max_length=192,                 # smaller = faster
    packing=False,

    per_device_train_batch_size=4,  # if OOM, set to 2
    gradient_accumulation_steps=4,  # effective batch 16
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=20,

    num_train_epochs=1,
    max_steps=100,                  # HARD CAP => ~10 min target

    eval_strategy="no",
    save_steps=100,
    logging_steps=10,
    save_total_limit=1,

    gradient_checkpointing=False,
    bf16=torch.cuda.is_available(),
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_small,
    eval_dataset=eval_small,
    args=training_args,
)

trainer.train()

Filter:   0%|          | 0/21600 [00:00<?, ? examples/s]

Filter:   0%|          | 0/21600 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/5000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/5000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/5000 [00:00<?, ? examples/s]

...

Tokenizing eval dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
10,42.971924
20,2.236118
30,1.563053
40,0.023881
50,0.616852
60,0.510663
70,0.235631
80,0.426156
90,0.182625
100,0.018354


TrainOutput(global_step=100, training_loss=4.878525710105896, metrics={'train_runtime': 1016.153, 'train_samples_per_second': 1.575, 'train_steps_per_second': 0.098, 'total_flos': 2449254069043200.0, 'train_loss': 4.878525710105896})

In [27]:
ADAPTER_DIR = "./adapter_abstain"

trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print("Saved adapter to:", ADAPTER_DIR)
print("Quantization mode used:", quant_mode)

Saved adapter to: ./adapter_abstain
Quantization mode used: 4bit


In [31]:
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR, use_fast=True)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Reload base (same as before)
try:
    base = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        device_map="auto",
        quantization_config=bnb_config,
    )
except Exception:
    base = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        device_map="auto",
        torch_dtype=torch.float16 if use_cuda else torch.float32,
    )

model = PeftModel.from_pretrained(base, ADAPTER_DIR)
model.eval()
print("Reloaded base + adapter.")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Reloaded base + adapter.


In [29]:
import re, string
import torch

@torch.no_grad()
def generate_tail(prompt: str, max_new_tokens=32):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,  # greedy
        temperature=1.0,
        pad_token_id=(tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id),
    )
    text = tokenizer.decode(out[0], skip_special_tokens=True)
    return text[len(prompt):].strip()

def is_abstain(s: str) -> bool:
    return normalize(s).startswith(normalize(ABSTAIN_STR))

def norm(s: str) -> str:
    s = (s or "").lower().strip()
    s = s.translate(str.maketrans("", "", string.punctuation))
    s = re.sub(r"\s+", " ", s)
    return s

def relaxed_match(pred: str, gold: str) -> bool:
    p, g = norm(pred), norm(gold)
    return p == g or p in g or g in p



sample_n = min(400, len(eval_ds))
sample = eval_ds.shuffle(seed=42).select(range(sample_n))

tp_ans = ans_total = 0
refuse_on_unans = unans_total = 0
over_refuse = hallucinate = 0

for ex in sample:
    prompt, gold_resp = split_text_to_prompt_resp(ex["text"])
    label = ex["label"]

    pred_resp = generate_tail(prompt)
    pred_is_abstain = is_abstain(pred_resp)

    if label == "answerable":
        ans_total += 1
        if pred_is_abstain:
            over_refuse += 1
        else:
            pred_final = extract_final_answer(pred_resp)
            gold_final = extract_final_answer(gold_resp)
            if relaxed_match(pred_final, gold_final):
                tp_ans += 1
    else:
        unans_total += 1
        if pred_is_abstain:
            refuse_on_unans += 1
        else:
            hallucinate += 1

print("\n==== Metrics (quick eval) ====")
print(f"Accuracy (answerable):        {tp_ans / max(1, ans_total):.3f}")
print(f"Refusal rate (unanswerable):  {refuse_on_unans / max(1, unans_total):.3f}")
print(f"Over-refusal (answerable):    {over_refuse / max(1, ans_total):.3f}")
print(f"Hallucination proxy (unans):  {hallucinate / max(1, unans_total):.3f}")
print(f"Eval samples used: {sample_n}")

The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


KeyboardInterrupt: 

In [32]:
#MoDIFIED FOR QUICKER EVAL
import re, string
import torch
from torch.utils.data import DataLoader

# ── batched generation ──────────────────────────────────────────────────────

EVAL_BATCH_SIZE = 16  # tune down to 8 if you hit OOM

@torch.no_grad()
def generate_batch(prompts: list[str], max_new_tokens=32) -> list[str]:
    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512,
    ).to(model.device)

    prompt_lengths = inputs["attention_mask"].sum(dim=1)  # track each prompt's length

    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=(tokenizer.pad_token_id if tokenizer.pad_token_id is not None
                      else tokenizer.eos_token_id),
    )

    results = []
    for i, seq in enumerate(out):
        # slice off the prompt tokens, decode only the new tokens
        new_tokens = seq[prompt_lengths[i]:]
        text = tokenizer.decode(new_tokens, skip_special_tokens=True)
        results.append(text.strip())
    return results


# ── helpers (unchanged) ─────────────────────────────────────────────────────

def is_abstain(s: str) -> bool:
    return normalize(s).startswith(normalize(ABSTAIN_STR))

def norm(s: str) -> str:
    s = (s or "").lower().strip()
    s = s.translate(str.maketrans("", "", string.punctuation))
    s = re.sub(r"\s+", " ", s)
    return s

def relaxed_match(pred: str, gold: str) -> bool:
    p, g = norm(pred), norm(gold)
    return p == g or p in g or g in p


# ── eval loop ───────────────────────────────────────────────────────────────

sample_n = min(400, len(eval_ds))
sample   = eval_ds.shuffle(seed=42).select(range(sample_n))

tp_ans = ans_total = 0
refuse_on_unans = unans_total = 0
over_refuse = hallucinate = 0

# chunk sample into batches
indices = list(range(sample_n))
batches = [indices[i:i + EVAL_BATCH_SIZE] for i in range(0, sample_n, EVAL_BATCH_SIZE)]

for batch_idx in batches:
    batch      = sample.select(batch_idx)
    prompts    = []
    gold_resps = []
    labels     = []

    for ex in batch:
        prompt, gold_resp = split_text_to_prompt_resp(ex["text"])
        prompts.append(prompt)
        gold_resps.append(gold_resp)
        labels.append(ex["label"])

    pred_resps = generate_batch(prompts)

    for pred_resp, gold_resp, label in zip(pred_resps, gold_resps, labels):
        pred_is_abstain = is_abstain(pred_resp)

        if label == "answerable":
            ans_total += 1
            if pred_is_abstain:
                over_refuse += 1
            else:
                pred_final = extract_final_answer(pred_resp)
                gold_final = extract_final_answer(gold_resp)
                if relaxed_match(pred_final, gold_final):
                    tp_ans += 1
        else:
            unans_total += 1
            if pred_is_abstain:
                refuse_on_unans += 1
            else:
                hallucinate += 1

print("\n==== Metrics (quick eval) ====")
print(f"Accuracy (answerable):        {tp_ans        / max(1, ans_total):.3f}")
print(f"Refusal rate (unanswerable):  {refuse_on_unans / max(1, unans_total):.3f}")
print(f"Over-refusal (answerable):    {over_refuse    / max(1, ans_total):.3f}")
print(f"Hallucination proxy (unans):  {hallucinate    / max(1, unans_total):.3f}")
print(f"Eval samples used: {sample_n}")


==== Metrics (quick eval) ====
Accuracy (answerable):        0.026
Refusal rate (unanswerable):  0.000
Over-refusal (answerable):    0.006
Hallucination proxy (unans):  1.000
Eval samples used: 400


In [34]:
prompt, gold_resp = split_text_to_prompt_resp(eval_ds[0]["text"])
pred_tail = generate_tail(prompt)
print("PRED TAIL:\n", pred_tail)
print("GOLD:\n", gold_resp)

PRED TAIL:
 FINAL: I don't know. I don't know. I don't know. I don't know. I don't know. I don't know.
GOLD:
 Answer:FINAL: BILLY COTTON


In [35]:
for i in range(5):
    ex = sample[i]
    prompt, gold_resp = split_text_to_prompt_resp(ex["text"])
    pred = generate_tail(prompt)
    print("\nLABEL:", ex["label"])
    print("PROMPT:\n", prompt[:300], "...")
    print("PRED:\n", pred)
    print("GOLD:\n", gold_resp)


LABEL: answerable
PROMPT:
 Context: ted to divination in 1791. The "reading tarots" based on the symbolic designs of the Tarot de Marseille (which were extensively modified to produce the widely known Rider-Waite deck) kept the older style of full-length character art, specific character meanings for the 21 trumps, and the us ...
PRED:
 FINAL: I don't know. I don't know. I don't know. I don't know. I don't know. I don't know.
GOLD:
 Answer:FINAL: Wild card

LABEL: unanswerable
PROMPT:
 Context: Pressing the sheet removes the water by force; once the water is forced from the sheet, a special kind of felt, which is not to be confused with the traditional one, is used to collect the water; whereas when making paper by hand, a blotter sheet is used instead.
Question:  What is used to  ...
PRED:
 FINAL: I don't know. I don't know. I don't know. I don't know. I don't know. I don't know.
GOLD:
 Answer:FINAL: I don't know.

LABEL: answerable
PROMPT:
 Context: "Disco Inferno" is a song by The 

In [36]:
print(eval_ds[0].keys())  # what fields exist?
print(eval_ds[0])

dict_keys(['prompt', 'response', 'label', 'text', 'completion'])
{'prompt': 'Context: "Somebody Stole My Gal" is a popular song from 1918, written by Leo Wood. In 1923, Ted Weems & his Orchestra had a five-week run at number one with his million-selling version.  It is also known in Japan as the theme song used by Yoshimoto Kogyo at its theater in Osaka.\n\nThe song has been featured in several Hollywood movies.  including:\n*Somebody Stole My Gal (1931)\n*Little Jack Little & Orchestra (1936)\n*When Willie Comes Marching Home (1950)\n*My Favorite Year (1982)\n*The Grass Harp (1995)\n*Melinda and Melinda (2004)\n*The Aviator (2004)\n\nOther recordings\n\n*Florence Millett - 1918\n*Ted Weems & His Orch. (Instr.) - 1924\n*Fletcher Henderson & His Orch. - 1924\n*Bix Beiderbecke - 1928\n*Fred Elizalde & His Anglo American Band - 1928\n*Bennie Moten\'s Kansas City Orch. - 1930\n*Ted Lewis & His Band (vocal: Ted Lewis)- 1931\n*Cab Calloway & His Orch. - 1931\n*Billy Cotton & His Band - 1931\

In [38]:
prompt, gold = split_text_to_prompt_resp(eval_ds[0]["text"])
print("PROMPT:", repr(prompt[-100:]))  # last 100 chars of prompt
print("GOLD:", repr(gold))

PROMPT: '899 to 1969 would you associate with the song ""Somebody Stole My Gal"" recorded in 1931 ?"\nAnswer:\n'
GOLD: 'FINAL: BILLY COTTON'
